# Record a dataset with a policy of your own

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccnets-team/causal-gpt-rl/blob/main/examples/record_dataset.ipynb)

The [quickstart](hub_quickstart.ipynb) loads a bundle and scores it. This one
runs a bundle to *produce* something: the episodes it drives, packaged as a
Minari dataset you can read back.

That is the second turn of the collection cycle. The first turn needs a policy
from somewhere else — SB3's PPO, a scripted controller, whatever already drives
your system — and ends at a dataset. After it, the policy is one this runtime
loads, and recording with it is the piece below.

Nothing here is Hopper-specific: the same cells run against any bundle and any
Gymnasium environment whose spaces it declares.

## Install

`mujoco` is pinned to `3.2.3` because that is the version the published datasets
were recorded with, and a different simulator release is a different measurement
even with identical weights and seeds.

`minari` is only needed by the last two cells. Recording needs neither it nor a
packaging environment.

In [ ]:
%pip install -q "causal-gpt-rl[hub,mujoco]" "mujoco==3.2.3" "minari==0.5.3"

## Get the repository

`collection/` is a directory of this repository, not part of the installed
package — the runtime is not supposed to start pulling in a dataset builder. On
Colab, clone it; in a checkout, this cell does nothing.

In [ ]:
from pathlib import Path

if not Path("collection").is_dir():
    !git clone -q https://github.com/ccnets-team/causal-gpt-rl.git
    %cd causal-gpt-rl

## Choose the bundle, and the retention

`kv_cache_max_len` is how much past the rollout keeps. It is a load-time
argument, not a per-step one, and it is the single parameter this cycle turns
on: a policy trained on a 32-step window can be run with far more history than
that.

Whether the extra history helps is environment-dependent — across the published
bundles it helps some and hurts others — so measure it in your environment
before spending a long collection run on it.

In [ ]:
import os

import torch

repo_id = "ccnets/causal-gpt-rl"
subfolder = "hopper-v5"
env_id = "Hopper-v5"

episodes = 3
max_steps = 1000
seed_start = 0
kv_cache_max_len = 256           # the bundle's own context_length is 32

raw_dir = Path("raw/hopper-v5")
dataset_id = "review/hopper-recorded-v0"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Keep the dataset beside this notebook rather than in ~/.minari.
os.environ["MINARI_DATASETS_PATH"] = str(Path("minari").resolve())

print(f"{repo_id}/{subfolder} -> {raw_dir} -> {dataset_id}  (device={device})")

## Load the policy

In [ ]:
import gymnasium as gym

from causal_gpt_rl.inference import load_runner_from_hub

env = gym.make(env_id)
policy = load_runner_from_hub(
    repo_id,
    subfolder=subfolder,
    device=device,
    kv_cache_max_len=kv_cache_max_len,
)
policy

## Record

`CollectionRunner` wraps the runner and keeps its calls — `reset` / `act` /
`observe`. Against the plain rollout loop the difference is the constructor
line, four arguments, and the disappearance of the `if not done` guard.

Three things it gets right that a hand-written loop usually does not:

- **Pairing.** The model's context token is `(state, previous action)`, one step
  off from what a dataset wants. The action written for a step is the one
  `act()` returned for the state it saw, taken at the API boundary rather than
  read back out of the model.
- **The terminal observation.** The runner has no use for the final state; the
  contract needs it, because `T` transitions need `T + 1` states. Inside the
  wrapper the split is invisible: that observation is recorded, not fed back.
- **Both ways an episode ends.** A flag closes it here; a `reset` arriving
  mid-episode drops the dangling action and closes it as a truncation.

Re-running this cell starts from a clean directory. Pass `resume=True` to the
constructor to add to one instead.

In [ ]:
import shutil

from collection import CollectionRunner

if raw_dir.exists():
    shutil.rmtree(raw_dir)

runner = CollectionRunner(policy, raw_dir, bundle=f"{repo_id}/{subfolder}")

for episode in range(episodes):
    obs, _ = env.reset(seed=seed_start + episode)
    runner.reset(obs, record=True)
    total = 0.0
    for step in range(max_steps):
        action = runner.act()
        obs, reward, terminated, truncated, _ = env.step(action)
        total += float(reward)
        # A terminal state is the stronger claim, so it clears truncation:
        # recording both would say two things about one transition.
        truncated = not terminated and (bool(truncated) or step + 1 == max_steps)
        runner.observe(obs, reward, terminated, truncated)
        if terminated or truncated:
            break
    ending = "terminated" if terminated else "truncated"
    print(f"[ep {episode}] return={total:.1f} steps={step + 1} {ending}")

runner.close()
env.close()

## What was written

One `ep_%06d.npz` per episode, and a `spec.json` declaring the spaces — read off
the bundle's own `observation_space` / `action_space`, so the dataset cannot
disagree with the policy that produced it.

`spec.json` also carries a provenance entry per recording run. The packager
ignores keys it does not know, and without these nothing afterwards can say
which policy, at which retention, wrote the episodes.

In [ ]:
import json

print(sorted(path.name for path in raw_dir.iterdir()))

spec = json.loads((raw_dir / "spec.json").read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in spec.items() if k != "provenance"}, indent=2))

kept = ("recorder", "causal_gpt_rl", "bundle", "context_length",
        "kv_cache_max_len", "bos_cache_mode")
print({k: spec["provenance"][0][k] for k in kept})

## Check one episode

The five arrays are the whole input contract. `observations` is one longer than
the rest, and exactly one episode boundary sits at the end.

In [ ]:
import numpy as np

with np.load(sorted(raw_dir.glob("ep_*.npz"))[0]) as episode:
    for name in ("observations", "actions", "rewards", "terminations", "truncations"):
        print(f"{name:14s} {episode[name].shape} {episode[name].dtype}")
    assert len(episode["observations"]) == len(episode["actions"]) + 1
    assert episode["terminations"][-1] or episode["truncations"][-1]
    print("ended by", "termination" if episode["terminations"][-1] else "truncation")

## Package

`build_dataset` preflights every episode before Minari creates anything, writes
in bounded batches, then loads the result back and verifies its counts and
spaces. It is the same code path as
`python collection/build_minari.py --raw raw/ --dataset-id ...`.

In [ ]:
import minari

from collection import build_dataset

if dataset_id in minari.list_local_datasets():
    minari.delete_dataset(dataset_id)   # only touches MINARI_DATASETS_PATH above

build_dataset(raw_dir, dataset_id, description="Recorded with CollectionRunner.")

## Read it back

A dataset that loads can still declare the wrong interface, so read the spaces
rather than trusting that packaging succeeded.

In [ ]:
dataset = minari.load_dataset(dataset_id)

print(dataset.total_episodes, "episodes /", dataset.total_steps, "transitions")
print("observations", dataset.observation_space)
print("actions     ", dataset.action_space)

dataset.set_seed(0)
sample = dataset.sample_episodes(n_episodes=1)[0]
print("sampled return", float(np.asarray(sample.rewards).sum()))

## What this shows, and what it does not

**It shows the mechanism, not a better dataset.** Three episodes off a strong
policy is a smoke run. The gain this cycle is after is a step up in the *tier*
of what you record, and it comes from retention above the trained window — which
helps some environments and hurts others.

**Check what ended your episodes.** If nothing terminated, the dataset has no
terminal states in it, and a termination head learns from it that nothing ever
ends. Raise `max_steps`, or record until real terminations appear.

**Packaging normally runs in its own `minari==0.5.3` environment.** One kernel
did both here because it can, not because it must; the recorder and the packager
share no dependency.

The script form of this notebook, for longer runs:

```
python -m examples.deploy.record --env-id Hopper-v5 --out raw/ \
    --episodes 20 --kv-cache-max-len 256
python collection/build_minari.py --raw raw/ --dataset-id review/hopper-v0
```

What the arrays have to contain, if you ever record from another source:
[the input contract](https://github.com/ccnets-team/causal-gpt-rl/blob/main/collection/docs/01-the-input-contract.md).